# Long Baseline Study of Mira Variable Stars: Gaia Part Three
## 24-in Observations Calibrated with Gaia
The majority of this notebook is taken from Emily's New Plate Analysis notebook, with adjustments made to calculate the magnitude in the end. Check each cell for multiple hashtags that indicate where you need to input your own info.

#### Step One
Use Emily's Image Calibration notebook to sort and calibrate your observation files. Retrieve WCS files of your observation through whatever astrometry.net route you can take.

#### Step Two
Do aperture photometry and calibrate with Gaia.

In [ ]:
#Start with imports!
import cv2
import numpy as np
from astropy.io import fits
from astroquery.gaia import Gaia
from astropy.coordinates.sky_coordinate import SkyCoord
from astropy.wcs import WCS
import sep
sep.set_extract_pixstack(1000000)
sep.set_sub_object_limit(1000000)

from astropy.table import Table, hstack
from scipy.optimize import curve_fit
import csv
import math 
from scipy.stats import mode as spmode
import statistics

import matplotlib.pyplot as plt
from astropy.visualization import SqrtStretch
from astropy.visualization.mpl_normalize import ImageNormalize
from astropy.stats import SigmaClip
from photutils.background import Background2D, MedianBackground
from photutils.background import *
from photutils.datasets import make_100gaussians_image
from astropy.stats import biweight_location
from astropy.stats import mad_std
from astropy.stats import sigma_clipped_stats

import astropy.table
from matplotlib.backends.backend_pdf import PdfPages

import numpy as np
from astropy.wcs import WCS
from astropy.table import Table, hstack

import os
from astropy.io import ascii
import glob
from glob import glob

from astroquery.astrometry_net import AstrometryNet

import glob

from datetime import datetime
from astropy.time import Time
import matplotlib.dates as mdates

%matplotlib widget

# sq func used in analysis 
def sq(x,a,b):
    return(a*x**(b/2))
import pandas as pd

mean = np.mean
std = np.std

In [ ]:
# def ap_phot2 detects and subtracts the image background through photutils and then extracts objects from image with SEP, calculating a mag

def ap_phot2(data, sigma):

    sigma_clip = SigmaClip(sigma = sigma)
    bkg_estimator = MMMBackground()

    bkg = Background2D(data,box_size=(msize, msize), sigma_clip=sigma_clip, bkg_estimator=bkg_estimator)
    data_sub = data - bkg.background

    objects = Table(sep.extract(data_sub, sigma, err = bkg.background_rms_median))

    ob_mask = np.where(objects['flag'] != 8)[0]
    objects = objects[ob_mask]

    objects['flux2'], sum_err, flag = sep.sum_circle(data=data_sub, x=objects['x'], y=objects['y'], r = ((objects['xmax'] -    objects['xmin'])/2), err=bkg.background_rms_median)

    return data_sub, objects

In [ ]:
first_file = '/raid6/users/rantoine/2026-07-24/lights/7-24-26/7-24-26_c/7-24-26_R_Aql_-001_60s_B_c.new'

# load the data
firstframe = fits.getdata(first_file)

# display the image
plt.figure(1) #Create new "figure" for this image
plt.imshow(firstframe,cmap='gray',vmin=np.percentile(firstframe,25),
           vmax=np.percentile(firstframe,99.9), origin = 'lower')

plt.show()

In [ ]:
path = '/raid6/users/rantoine/2026-07-24/lights/7-24-26/7-24-26_c/'

filename = '7-24-26_R_Aql_-004_60s_B_c.new'

matches = sorted(glob.glob(path + filename))

msize=50

####### CHANGE TARGET LOCATION #######

tx, ty = (391, 581) #find using matplotlib widget of the image you displayed above

err = 15 #tolerance in pixel distance

fluxes = []

for filepath in matches:
    with fits.open(filepath) as hdu:
        data = hdu[0].data
        header = hdu[0].header

    data_sub, objects = ap_phot2(data, 5)

    df = objects.to_pandas()

    match = df[
        (df['x'].sub(tx).abs() <= err) &
        (df['y'].sub(ty).abs() <= err)
    ]

    print(f"\n{filepath}")
    if not match.empty:
        with pd.option_context('display.max_rows', None, 'display.max_columns', None):
            display(match)

        fluxes.append(match['flux'])
        
    else:
        norm = ImageNormalize(stretch=SqrtStretch())   #to check where your star is and increase error tolerance
        plt.figure()
        plt.imshow(data, origin='lower', cmap='Greys_r', norm=norm, interpolation='nearest')
        plt.show()
  
fluxes

####### CHANGE FILE NAME FOR EACH DATE OF OBSERVATION #######

#np.save('7_24_26_R_Aql_fluxes1.npy', fluxes)

#### Step Three
Repeat for each night of observations.

#### Step Four
Combine your observations and calculate the magnitude.

In [ ]:
flux_files = sorted(glob.glob('/raid6/users/rantoine/Mira_Notebooks/Better_Part_Three/*R_Aql_fluxes1.npy'))

dates = []
avg_flux = []
for filepath in flux_files:
    filename = os.path.basename(filepath)
    date_str = filename.split('_R_Aql')[0] #taking the date from the filename
    date_time = datetime.strptime(date_str, "%m_%d_%y")
    #date = Time(date_time, scale="utc")
    
    df = np.load(filepath)

    flux = df[0]
    
    avg_flux.append(np.mean(flux))
    dates.append(date_time)
print(avg_flux)
print(dates)

df = pd.DataFrame({'Date': pd.to_datetime(dates), 'Avg_Flux': avg_flux})

In [ ]:
# Now, let's plot our flux lightcurve!
plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 13,
    'figure.titlesize': 18
})

fig, ax = plt.subplots(figsize=(10, 5))

ax.scatter(df['Date'], df['Avg_Flux'], color='#9C179E', s=50)
ax.set_xlabel('Time (Date)')
ax.set_ylabel('Flux')
ax.invert_yaxis()
ax.set_title('Yerkes Flux Light Curve of R Aql')

fig.autofmt_xdate()

In [ ]:
plt.close()

#### Testing next steps

In [ ]:
def gaia(header):
    wcs_header = WCS(header)
    bottom_coord = wcs_header.pixel_to_world(0, 0)
    top_coord = wcs_header.pixel_to_world(header['NAXIS1'], header['NAXIS2'])
    full_coord_ra = np.abs(top_coord.ra.deg-bottom_coord.ra.deg)
    full_coord_dec = np.abs(top_coord.dec.deg-bottom_coord.dec.deg)
    ra_center = (top_coord.ra.deg + bottom_coord.ra.deg) / 2
    dec_center = (top_coord.dec.deg + bottom_coord.dec.deg) / 2
    job = Gaia.launch_job("SELECT TOP 300000 "
                    "source_id,ra,dec,parallax,parallax_error,pm,pmra,pmra_error,pmdec,pmdec_error,"
                    "phot_g_mean_mag, phot_g_mean_flux, phot_bp_mean_mag,phot_bp_mean_flux,phot_rp_mean_mag,"
                    "phot_rp_mean_flux,bp_rp, phot_variable_flag, classprob_dsc_combmod_galaxy"
                    " from gaiadr3.gaia_source"
                    " WHERE CONTAINS(POINT('ICRS',ra,dec),BOX('ICRS',{0},{1},{2},{3}))=1 AND"
                    "((phot_bp_mean_mag + 0.9*bp_rp) <= 19.0)".format(ra_center,
                                                                      dec_center,
                                                                    full_coord_ra, full_coord_dec))
    gaia_res = job.get_results()
    gaia_coord = SkyCoord(gaia_res['ra'], gaia_res['dec'], frame = 'icrs', unit = 'deg')
    
    return gaia_res, gaia_coord

def make_plots(tablenum, sigma):
    pg_list = list(tablenum['pg'])
    mag_list = list(tablenum['sep_mag'])
    newpglist = []
    newmaglist = []
    for i in range(len(pg_list)):
        newpglist.append(round(pg_list[i],2))
        newmaglist.append(round(mag_list[i],2))
    
    diff = spmode(newpglist)[0] - spmode(newmaglist)[0]
    
    diff_factor = diff
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3)
    fig.set_figheight(6)
    fig.set_figwidth(24)
    
    ax1.set_title('Sigma: %d, %d stars found! '%(sigma, len(tablenum)))
    ax1.scatter(tablenum['pg'],tablenum['sep_mag'] + diff_factor, s = 3, alpha = 0.2)
    ax1.plot(np.arange(min(tablenum['pg']),max(tablenum['pg'])+2),np.arange(min(tablenum['pg']),max(tablenum['pg'])+2), color = 'r', ls = '--', alpha = 0.5)
    ax1.scatter(spmode(newpglist)[0], spmode(newmaglist)[0]+ diff_factor)    
    ax1.set_xlabel('Phot Mag Approx from Gaia')
    ax1.set_ylabel('Phot Mag Approx from SEP')
    
    
    
    
    ax2.scatter(tablenum['pg'],tablenum['pg'] - tablenum['sep_mag']- diff_factor, s = 3, alpha = 0.2)
    ax2.set_title('Sigma: %d, %d stars found! '%(sigma, len(tablenum)))
    ax2.hlines(0,min(tablenum['pg']),max(tablenum['pg'])+2, color='r', ls = '--', alpha = 0.5)
    ax2.set_xlabel('Phot Mag Approx from Gaia')
    ax2.set_ylabel('Residual of Phot Mag Approx from SEP and Gaia')
    
    

    ax3.scatter(tablenum['x'],tablenum['pg'] - tablenum['sep_mag']- diff_factor, s = 3, alpha = 0.2)
    ax3.set_title('Sigma: %d, %d stars found! '%(sigma, len(tablenum)))
    ax3.set_xlabel('X Coordinate')
    ax3.set_ylabel('Residual of Phot Mag Approx from SEP and Gaia')

    plt.show()


    print('Phot Mag Approx from Gaia: ' + str(mean(tablenum['pg'])))
    print('\nPhot Mag Approx from SEP: ' + str(mean(tablenum['sep_mag']) + diff_factor))
    print('\nResidual of Phot Mag Approx from SEP and Gaia: ' + str(mean(tablenum['pg'] - (tablenum['sep_mag'])- diff_factor) ))
    print('STD Residual: ' + str(std(tablenum['pg'] - (tablenum['sep_mag'])) ))

    
def alpha_finder(data_sub, wcs, otab, gtab, gcor):

    a_list = list(np.round(np.arange(-0.5,1.5,0.1), 2))
    avglist = []
    avglist2 = []

    for a, alpha_val in enumerate(a_list):
        #print('Alpha Value: ' + str(alpha_val))
        alpha = alpha_val
        
        # photographic magnitude approximation
        gtab['pg'] = gtab['phot_bp_mean_mag']+ alpha * gtab['bp_rp']
        gtab['pg_flux'] = gtab['phot_bp_mean_flux']+ alpha *(gtab['phot_bp_mean_flux']
                                                                 /gtab['phot_rp_mean_flux'])
        
    
        #match of gaia and s
        mtab = match_table(otab,wcs,gtab,gcor)

        scal = np.max(mtab['flux2'])/np.max(data_sub) 
        popt, ___ = curve_fit(sq, mtab['flux2']/scal, mtab['pg_flux']/np.pi)    
        ndata = sq(data_sub,*popt)    
        
        
        avglist.append(mean(mtab['pg'] - ((mtab['sep_mag']))))
        pg_list = list(mtab['pg'])
        mag_list = list(mtab['sep_mag'])
        newpglist = []
        newmaglist = []
        for i in range(len(pg_list)):
            newpglist.append(np.round(pg_list[i],3))
            newmaglist.append(np.round(mag_list[i],3))
    
    
        
        diff = spmode(newpglist)[0] - spmode(newmaglist)[0]
        avglist2.append(mean(mtab['pg'] - ((mtab['sep_mag']+diff))))
    
    
    
    avgmin = min(np.abs(avglist))
    index = np.where(np.abs(avglist) == avgmin)[0][0]
    alpha1 = a_list[index]
    
    
    avgmin2 = min(np.abs(avglist2))
    index2 = np.where(np.abs(avglist2) == avgmin2)[0][0]
    alpha2 = a_list[index2]

    print('Alpha 1: %.1f and Alpha 2: %.1f'%(alpha1, alpha2))

    return alpha1, alpha2


def match_table(object_table, wcs, gaia_table, gaia_coordinates):

    pcor = wcs.pixel_to_world(object_table['x'], object_table['y'])
    object_table['ra_p'], object_table['dec_p'] = pcor.ra.deg, pcor.dec.deg
    index, d2d, ____ = pcor.match_to_catalog_sky(gaia_coordinates)
    mtab = hstack([gaia_table[index], Table(object_table)])
    _, unique_ind = np.unique(mtab['source_id'], return_index=True)
    mtab = mtab[unique_ind]
    
    mtab['ang_dist'] = np.abs(np.sqrt(mtab['ra']**2+mtab['dec']**2)
                              -np.sqrt(mtab['ra_p']**2+mtab['dec_p']**2))
    mtab['dec_res'] = (mtab['dec'] -  mtab['dec_p'])*3600
    mtab['ra_res'] = (mtab['ra'] -  mtab['ra_p'])*3600*0.9114
    
            
    return mtab

    
# def analysis 1 and 2 call all of our functions to create a table and alter our data 
def analysis_pt1(path, filename, sigma, msize):

    with fits.open(path + filename) as hdu:
        data = hdu[0].data.astype(np.float32)
        header = hdu[0].header
        #header['EXPTIME'] = (3600.0, 'Exposure time in seconds')
        #header['TSCOPE'] = 'Y41'
        #header['OBSDATE'] = ('1979-07-02', 'YYYY-MM-DD')
        #header['EMULSION'] = '103a-E'
        #header['PLTSIZE'] = ('3.25x4.25', 'Plate size in inches')
        
    wcs = WCS(header)

    #gaia querey results 
    gtab, gcor = gaia(header)
    
    
    #objects from scan 
    data_sub, otab = ap_phot2(data, sigma)

    alpha1, alpha2 = alpha_finder(data_sub, wcs, otab, gtab, gcor)

    print('Using alpha: ' + str(alpha2))

    # photographic magnitude approximation
    gtab['pg'] = gtab['phot_bp_mean_mag']+ alpha2 * gtab['bp_rp']
    gtab['pg_flux'] = gtab['phot_bp_mean_flux']+ alpha2 *(gtab['phot_bp_mean_flux']
                                                             /gtab['phot_rp_mean_flux'])
    
    #match of gaia and scan
    mtab = match_table(otab,wcs,gtab,gcor)
    
    scal = np.max(mtab['flux2'])/np.max(data_sub) 
    
    popt, ___ = curve_fit(sq, mtab['flux2']/scal, mtab['pg_flux']/np.pi) 
    ndata = sq(data_sub,*popt) 

    
    return mtab, ndata, header, data_sub, gtab, gcor



def analysis_pt2(ndata, sigma, gtab, gcor):
    wcs = WCS(header)
    data_sub, otab = ap_phot2(ndata,  sigma)
    mtab = match_table(otab, wcs, gtab, gcor)
    return mtab, data_sub






In [ ]:
# runs part one of analysis

%matplotlib ipympl
mtab1, ndata, header, data_sub, gtab, gcor= analysis_pt1(path, filename, 5, msize)
make_plots(mtab1, 5)